# 6. PPT — Reporte Diario

Este notebook **declara** lo que necesita y arma la presentación. No calcula indicadores:
consume el universo cacheado que produce `src/pipeline`.

Para crear otra PPT (notebooks 8, 9, …) se copia esta estructura y se cambian dos cosas:
el **perfil** y la lista de **rebanadas requeridas**. Si el perfil comparte la capa A con
uno ya ejecutado, el matching no se vuelve a correr.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.append('../1_funciones')

from datetime import date
from pathlib import Path

from IPython.display import Image, display
from IPython.utils.capture import capture_output

import pandas as pd
import numpy as np
import unicodedata

from src.settings import Settings
from src.pipeline import Universo
from src.slices import Slice
from src.ppt_engine import PPTReportBuilder, dimensiones_placeholder
from src.utils import (agrupar_por_mes, cruzar_mj_rips,
                       diferencias_vs_corte, deltas_vs_corte,
                       formatear_fecha_es)
from src.cortes_referencia import bajas_vs_corte, movimientos_vs_corte
from src.directorio_indicios import indicios_formalizados
from src.particiones import porcentajes_particion
from src.exportacion_fechada import exportar_con_fecha
from src.graficas_sin_datos import sin_registros, tolerar_sin_datos

from barras_verticales import barras_verticales
from barras_horizontales import barras_horizontales

## 1. Configuración de esta PPT

Un perfil propio (`perfiles/reporte_diario.toml`). Cambiar aquí de perfil cambia la
configuración de todo el pipeline sin tocar `config.py` ni el notebook 4.

In [ ]:
import importlib

import config

# `config` se recarga en cada corrida: un kernel abierto desde antes de editarlo
# conserva la versión vieja en memoria, y una constante recién añadida falla con
# AttributeError aunque esté en el archivo. Recargar evita tener que reiniciar el
# kernel cada vez que se toca config.py.
importlib.reload(config)

PERFIL = "reporte_diario"

# La plantilla se declara aquí, no en el armado: la sección 4 la necesita antes,
# para generar cada gráfica con las medidas de su cuadro de texto.
PLANTILLA = config.PLANTILLA_REPORTE_DIARIO
# El nombre lleva la fecha de HOY, no la del corte de los datos: cada emisión
# deja su propio archivo y no pisa la del día anterior. Se recompone aquí y no
# se toma `config.PPT_DIARIO_FILE` tal cual, porque esa constante se congela con
# la fecha en que se importó config: un kernel abierto de un día para otro
# seguiría escribiendo con la fecha de ayer.
SALIDA    = config.ppt_diario_file()

# La plantilla es un insumo: si falta, no hay nada que armar.
if not PLANTILLA.exists():
    raise FileNotFoundError(
        f"\u274c No se encontró la plantilla del reporte diario.\n"
        f"   Ruta declarada en config.PLANTILLA_REPORTE_DIARIO: {PLANTILLA}"
    )


cfg = Settings.desde_perfil(PERFIL)
print(cfg.resumen())

## 2. Rebanadas requeridas

**Sólo se calcula lo que está en esta lista.** Agregar una rebanada cuesta microsegundos;
ninguna obliga a recalcular el matching.

Dimensiones disponibles:

- `base`: `rn`, `ba`, `rn-no-ba`, `rn-si-ba`, `ba-no-rn`, `ba-si-rn`
- `periodo`: llaves de `PERIODOS_TIEMPO` (`01oct24-max`, `min-max`, …)
- `estatus`: `des`, `loc`, `all`
- `subfiltro`: `cdel`, `sdel`, `cind`, `sind`, `dfor`, `cpar`, `sisf`, `cisf`, `ccurp`, `scurp`, `cfval`, `sfval`, `cfvac`, `sfvac`, `fis`, `com`, `por`, `cvid`, `svid`
- `estado`: clave INEGI o `None` para nacional

In [ ]:
# Marcadores {{...}} que pide input/plantillas/reporte_diario.pptx.
# Se calculan SÓLO éstos; agregar uno cuesta microsegundos.
# Marcadores {{...}} de input/plantillas/reporte_diario.pptx.
# Se calcula SÓLO lo declarado aquí; agregar una rebanada cuesta microsegundos.

CONTEOS = [
    # ── Universo RNPDNO ──────────────────────────────────────────────────
    Slice("rn", "min-max"),
    Slice("rn", "min-max", estatus="des"),
    Slice("rn", "min-max", estatus="loc"),
    # ── Localizados históricos: por delito y por estado de vida ──────────
    Slice("rn", "min-max", estatus="loc", subfiltro="cdel"),
    Slice("rn", "min-max", estatus="loc", subfiltro="sdel"),
    Slice("rn", "min-max", estatus="loc", subfiltro="cvid"),
    Slice("rn", "min-max", estatus="loc", subfiltro="svid"),
    Slice("rn", "01oct24-max"),
    Slice("rn", "01oct24-max", estatus="des"),
    Slice("rn", "01oct24-max", estatus="loc"),
    # ── Localizados del sexenio: por delito y por estado de vida ─────────
    Slice("rn", "01oct24-max", estatus="loc", subfiltro="cdel"),
    Slice("rn", "01oct24-max", estatus="loc", subfiltro="sdel"),
    Slice("rn", "01oct24-max", estatus="loc", subfiltro="cvid"),
    Slice("rn", "01oct24-max", estatus="loc", subfiltro="svid"),
    # ── Corte de 2006 (lámina 2: "larga data" vs "2006 - 2026") ─────────
    # La plantilla rotula estos dos como "Larga data 1952 a 2005" y
    # "Registros Estatus desaparecido 2006 - 2026", así que el corte es 2006.
    # Los marcadores dicen 01ene26; hay que renombrarlos a 01ene06.
    Slice("rn", "min-01ene06", estatus="des"),   # larga data: 2,404
    Slice("rn", "01ene06-max", estatus="des"),   # 2006-2026: 131,782
    # Sin información suficiente: sin CURP válida y sin registro en PUI.
    # La bandera se llamaba `is_sdat`; se renombró a `is_sisf` para que coincida
    # con los marcadores de la plantilla y no se confunda con `scurp`/`smuni`.
    Slice("rn", "01ene06-max", estatus="des", subfiltro="sisf"),   # 46,768
    # `cisf` es el complemento de `sisf`, y cind/sind lo parten en dos ramas
    # que suman cisf: cisf_cind + cisf_sind = cisf.
    Slice("rn", "01ene06-max", estatus="des", subfiltro="cisf"),
    Slice("rn", "01ene06-max", estatus="des", subfiltro="cisf_cind"),
    Slice("rn", "01ene06-max", estatus="des", subfiltro="cisf_sind"),
    # Las mismas tres ramas, pero sólo del sexenio (lámina 9): son los recuadros
    # junto a la gráfica `gr_mes_rn_01oct24-max_des_sisf-cind-sind`.
    #     sisf + cisf_cind + cisf_sind = reg_rn_01oct24-max_des
    Slice("rn", "01oct24-max", estatus="des", subfiltro="sisf"),
    Slice("rn", "01oct24-max", estatus="des", subfiltro="cisf_cind"),
    Slice("rn", "01oct24-max", estatus="des", subfiltro="cisf_sind"),
    # ── Cruce con BANCI ──────────────────────────────────────────────────
    Slice("rn-si-ba", "01oct24-max", estatus="des"),
    Slice("rn-si-ba", "01oct24-max", estatus="des", subfiltro="cpar"),
    Slice("rn-si-ba", "01oct24-max", estatus="des", subfiltro="dfor"),
    # ── Sin match en BANCI, por folio válido ─────────────────────────────
    Slice("rn-no-ba", "01oct24-max", estatus="des"),
    Slice("rn-no-ba", "01oct24-max",     estatus="des", subfiltro="cfval"),
    # `sfval` completo: el complemento de `cfval` dentro de los sin match, o sea
    # "Sin carpeta de investigación" de las láminas 5 y 10. Es la suma de las dos
    # ramas de abajo: sfval_sfvac + sfval_cfvac = sfval.
    Slice("rn-no-ba", "01oct24-max",     estatus="des", subfiltro="sfval"),
    Slice("rn-no-ba", "01oct24-max",     estatus="des", subfiltro="sfval_sfvac"),
    # Corte de agosto 2025 dentro de cada rama de folio (lámina 4)
    Slice("rn-no-ba", "01oct24-01ago25", estatus="des", subfiltro="cfval"),
    Slice("rn-no-ba", "01ago25-max",     estatus="des", subfiltro="cfval"),
    Slice("rn-no-ba", "01oct24-01ago25", estatus="des", subfiltro="sfval_sfvac"),
    Slice("rn-no-ba", "01ago25-max",     estatus="des", subfiltro="sfval_sfvac"),
    # Corte de agosto 2026 dentro de cada rama de folio (láminas 13 y 14). Es
    # otro corte, no un reemplazo: el de 2025 sigue partiendo la lámina 5.
    Slice("rn-no-ba", "01ago26-max",     estatus="des", subfiltro="cfval"),
    Slice("rn-no-ba", "01ago26-max",     estatus="des", subfiltro="sfval_sfvac"),
    # Institución que originó el registro. `is_por` es el residuo: ni fiscalía
    # ni comisión, así que fis + com + por = la rama de folio completa.
    # Son los tres recuadros laterales de las láminas por entidad: el sexenio
    # completo en las 11 y 12, el corte desde agosto 2026 en las 13 y 14.
    *[Slice("rn-no-ba", periodo, estatus="des", subfiltro=f"{folio}_{inst}")
      for periodo in ("01oct24-max", "01ago26-max")
      for folio in ("cfval", "sfval_sfvac")
      for inst in ("fis", "com", "por")],
    # Campo de folio vacío: la otra rama de `sfval`, que ya no es hermana de
    # `cfval` sino su subdivisión:
    #     cfval + sfval = ..._des   y   sfval_sfvac + sfval_cfvac = sfval
    Slice("rn-no-ba", "01oct24-max",     estatus="des", subfiltro="sfval_cfvac"),
    Slice("rn-no-ba", "01oct24-01ago25", estatus="des", subfiltro="sfval_cfvac"),
    Slice("rn-no-ba", "01ago25-max",     estatus="des", subfiltro="sfval_cfvac"),
]

# Porcentajes: cada uno se calcula contra su denominador `Slice.padre`, es decir
# la misma rebanada sin su último filtro. Emiten la clave `pct_<nombre>`.
PORCENTAJES = [
    # Cada entrada es un Slice (denominador = su padre) o una tupla
    # (Slice, denominador) cuando la regla del padre no da la lectura correcta.

    # ── Regla del padre: contra el nivel inmediato superior ──────────────
    Slice("rn", "min-max", estatus="des"),                                   # / reg_rn_min-max
    Slice("rn", "min-max", estatus="loc"),                                   # / reg_rn_min-max
    Slice("rn", "01oct24-max", estatus="des"),                               # / reg_rn_01oct24-max
    Slice("rn", "01oct24-max", estatus="loc"),                               # / reg_rn_01oct24-max
    Slice("rn", "01oct24-max", estatus="loc", subfiltro="cdel"),             # / ..._loc
    Slice("rn", "01oct24-max", estatus="loc", subfiltro="sdel"),             # / ..._loc
    Slice("rn", "01oct24-max", estatus="loc", subfiltro="cvid"),             # / ..._loc
    Slice("rn", "01oct24-max", estatus="loc", subfiltro="svid"),             # / ..._loc
    Slice("rn", "min-max", estatus="loc", subfiltro="cdel"),                 # / reg_rn_min-max_loc
    Slice("rn", "min-max", estatus="loc", subfiltro="sdel"),                 # / reg_rn_min-max_loc
    Slice("rn", "min-max", estatus="loc", subfiltro="cvid"),                 # / reg_rn_min-max_loc
    Slice("rn", "min-max", estatus="loc", subfiltro="svid"),                 # / reg_rn_min-max_loc
    Slice("rn", "01ene06-max", estatus="des", subfiltro="sisf"),             # / reg_rn_01ene06-max_des
    Slice("rn", "01ene06-max", estatus="des", subfiltro="cisf"),             # / reg_rn_01ene06-max_des
    Slice("rn-si-ba", "01oct24-max", estatus="des", subfiltro="cpar"),       # / ..._des
    Slice("rn-si-ba", "01oct24-max", estatus="des", subfiltro="dfor"),       # / ..._des

    # ── Denominador explícito · las tres ramas de información suficiente ─
    # La regla del padre mediría cisf_cind y cisf_sind contra `..._des_cisf`.
    # La lámina las presenta junto a `sisf` como reparto del total 2006-max,
    # así que las tres se miden contra los desaparecidos de ese período.
    *[(Slice("rn", "01ene06-max", estatus="des", subfiltro=sub),
       Slice("rn", "01ene06-max", estatus="des"))
      for sub in ("cisf_cind", "cisf_sind")],

    # ── Denominador explícito · antes + después de Ago 2025 = 100% ───────
    # Cada corte se reparte el total de SU rama de folio, no el total de su período.
    *[(Slice("rn-no-ba", periodo, estatus="des", subfiltro=folio),
       Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro=folio))
      for folio in ("cfval", "sfval_sfvac", "sfval_cfvac")
      for periodo in ("01oct24-01ago25", "01ago25-max")],
]

# ── Porcentajes que suman exactamente 100% ──────────────────────────────────
# Las láminas 5, 6 y 7 reparten a los desaparecidos del sexenio en recuadros, y
# cada reparto debe cerrar en 100.0%. Redondear cada porcentaje por su lado (lo
# que hace PORCENTAJES) puede dejarlo en 99.9% o 100.1%, así que éstos salen de
# `porcentajes_particion`, que reparte las décimas por resto mayor.
#
# La misma rebanada aparece en varias láminas contra denominadores distintos, y
# el sufijo del marcador dice a qué reparto pertenece. Dentro de una lámina, los
# que llevan el mismo sufijo suman 100%:
#     _v1  láminas 5 y 7:  si-ba + no-ba                      = rn_..._des
#          lámina 6:       sfval_sfvac + sfval_cfvac          = no-ba_..._des_sfval
#     _v2  lámina 6:       si-ba + no-ba_cfval + no-ba_sfval  = rn_..._des
#     _v3  lámina 7:       cfval + sfval_sfvac + sfval_cfvac  = no-ba_..._des
# `si-ba` no se mide contra su padre: `rn-si-ba_01oct24-max` es idéntico a su
# numerador —el matching sólo alcanza desaparecidos del sexenio— y daría 100%.
#
# Cada entrada es (sufijo, denominador, partes). Si las partes no suman el
# denominador, el cálculo se detiene en vez de forzar el 100%.
DEN_DES_SEXENIO = Slice("rn", "01oct24-max", estatus="des")
DEN_SIN_MATCH   = Slice("rn-no-ba", "01oct24-max", estatus="des")
DEN_SFVAL       = Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval")

PARTICIONES = [
    ("_v1", DEN_DES_SEXENIO, [Slice("rn-si-ba", "01oct24-max", estatus="des"),
                              Slice("rn-no-ba", "01oct24-max", estatus="des")]),
    ("_v1", DEN_SFVAL,       [Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro=sub)
                              for sub in ("sfval_sfvac", "sfval_cfvac")]),
    ("_v2", DEN_DES_SEXENIO, [Slice("rn-si-ba", "01oct24-max", estatus="des"),
                              *[Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro=sub)
                                for sub in ("cfval", "sfval")]]),
    ("_v3", DEN_SIN_MATCH,   [Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro=sub)
                              for sub in ("cfval", "sfval_sfvac", "sfval_cfvac")]),
]

# PENDIENTE: 4 marcadores usan `ccvac`/`scvac` (campos vacíos), cuya definición
# sigue sin decidirse, así que no se calculan (son los únicos que faltan):
#     reg_rn-no-ba_01oct24-max_des_sfval_ccvac
#     reg_rn-no-ba_01oct24-max_des_sfval_scvac
#     pct_reg_rn-no-ba_01oct24-max_des_sfval_ccvac
#     pct_reg_rn-no-ba_01oct24-max_des_sfval_scvac
# Cuando se defina, basta con implementar is_ccvac / is_scvac en
# src/pipeline/derivadas.py, subir VERSION_DERIVADAS y añadirlas a estas listas.

# DataFrames que necesitan las gráficas (no todos los indicadores requieren uno).
PARA_GRAFICAS = [
    Slice("rn-si-ba", "01oct24-max", estatus="des"),
]

# DataFrames que además se escriben a disco como Parquet, uno por rebanada,
# en cfg.resultados_dir con el nombre canónico del Slice:
#     output/Resultados_4_Generacion_Indicadores/reg_rn_01oct24-max_des.parquet
#     output/Resultados_4_Generacion_Indicadores/reg_rn_min-max.parquet
# Es la entrega para R/Excel; el consumo desde Python no los necesita.
PARA_EXPORTAR = [
    Slice("rn", "01oct24-max", estatus="des"),   # reg_rn_01oct24-max_des
    Slice("rn", "min-max"),                      # reg_rn_min-max
    # Las tres ramas de información suficiente del período 2006-max. Se entregan
    # como base, no sólo como conteo: son las que se comparan contra el corte
    # congelado del 25/03/2026 (ver DIFERENCIAS_VS_MARZO más abajo), y quien
    # revisa la diferencia necesita los registros, no el número. Las mismas tres
    # se comparan también contra el corte del 01/07/2026 (DIFERENCIAS_VS_JULIO).
    #     sisf + cisf_cind + cisf_sind = reg_rn_01ene06-max_des
    Slice("rn", "01ene06-max", estatus="des", subfiltro="sisf"),       # reg_rn_01ene06-max_des_sisf
    Slice("rn", "01ene06-max", estatus="des", subfiltro="cisf_cind"),  # reg_rn_01ene06-max_des_cisf_cind
    Slice("rn", "01ene06-max", estatus="des", subfiltro="cisf_sind"),  # reg_rn_01ene06-max_des_cisf_sind
]

# Bases que se entregan fuera, con la fecha de HOY en el nombre (no la del corte)
# y en Parquet y Excel. Cada día deja su propio par de archivos y no pisa el
# anterior, a diferencia de PARA_EXPORTAR:
#     output/Resultados_4_Generacion_Indicadores/reg_rn-no-ba_01jul26-max_des_sfval_08102026.parquet
#     output/Resultados_4_Generacion_Indicadores/reg_rn-no-ba_01jul26-max_des_sfval_08102026.xlsx
PARA_EXPORTAR_CON_FECHA = [
    # Desaparecidos sin match en BANCI y sin carpeta válida desde julio de 2026:
    # la base de la revisión manual de carpetas. Lo que esa revisión dé por
    # válido vuelve por input/carpetas_manuales/ y sale de aquí en la siguiente.
    Slice("rn-no-ba", "01jul26-max", estatus="des", subfiltro="sfval"),   # reg_rn-no-ba_01jul26-max_des_sfval
]
# Marcadores de diferencia `<actual>-<corte>`: cuánto creció el indicador desde
# el corte congelado del 25/03/2026 (`input/datos_marzo`, ver
# config.CORTE_25MAR26_FILES). El corte no pasa por el pipeline —llega como
# Excel ya rebanado—, así que se cuenta aparte y se resta.
DIFERENCIAS_VS_MARZO = {
    "reg_rn_01ene06-max_des_sisf":      "reg_rn_25mar26_des_sisf",
    "reg_rn_01ene06-max_des_cisf_cind": "reg_rn_25mar26_des_cisf_cind",
    "reg_rn_01ene06-max_des_cisf_sind": "reg_rn_25mar26_des_cisf_sind",
}

# Marcadores `_ddes` / `_dloc`: en vez de restar dos totales, cruzan los `fub`
# del indicador de hoy contra los del corte de marzo para separar las altas de
# las bajas. `ddes` − `dloc` reproduce la diferencia declarada arriba.
DELTAS_VS_MARZO = {
    Slice("rn", "01ene06-max", estatus="des", subfiltro="sisf"):
        "reg_rn_25mar26_des_sisf",
    Slice("rn", "01ene06-max", estatus="des", subfiltro="cisf_cind"):
        "reg_rn_25mar26_des_cisf_cind",
    Slice("rn", "01ene06-max", estatus="des", subfiltro="cisf_sind"):
        "reg_rn_25mar26_des_cisf_sind",
}

# ── Lámina 4 · los mismos marcadores contra el corte del 01/07/2026 ─────────
# Segunda foto congelada (`input/datos_julio`, ver config.CORTE_01JUL26_FILES),
# con las mismas tres rebanadas que la de marzo. Las diferencias llevan el corte
# en el nombre (`...-reg_rn_01jul26_des_sisf`), así que no chocan con las de
# marzo. Los `_ddes` / `_dloc` no lo llevan: la plantilla los distingue con
# SUFIJO_DELTAS_VS_JULIO (`reg_rn_01ene06-max_des_sisf_ddes_v2`).
DIFERENCIAS_VS_JULIO = {
    "reg_rn_01ene06-max_des_sisf":      "reg_rn_01jul26_des_sisf",
    "reg_rn_01ene06-max_des_cisf_cind": "reg_rn_01jul26_des_cisf_cind",
    "reg_rn_01ene06-max_des_cisf_sind": "reg_rn_01jul26_des_cisf_sind",
}
DELTAS_VS_JULIO = {
    Slice("rn", "01ene06-max", estatus="des", subfiltro="sisf"):
        "reg_rn_01jul26_des_sisf",
    Slice("rn", "01ene06-max", estatus="des", subfiltro="cisf_cind"):
        "reg_rn_01jul26_des_cisf_cind",
    Slice("rn", "01ene06-max", estatus="des", subfiltro="cisf_sind"):
        "reg_rn_01jul26_des_cisf_sind",
}
SUFIJO_DELTAS_VS_JULIO = "_v2"

# Bases detrás de los marcadores `_dloc`: las filas del corte de marzo que hoy ya
# no están en el indicador, con el estado que hoy tiene cada una en el RNPDNO.
# `_dloc` es una cota superior de las localizaciones —un registro también sale
# si dejó de cumplir el subfiltro—, y quien la revisa necesita los registros para
# separar los casos. Cada una se cruza contra el corte que le asigna
# DELTAS_VS_MARZO y se escribe junto a las de PARA_EXPORTAR:
#     output/Resultados_4_Generacion_Indicadores/reg_rn_01ene06-max_des_sisf_dloc.parquet
BAJAS_A_EXPORTAR = [
    Slice("rn", "01ene06-max", estatus="des", subfiltro="sisf"),   # reg_rn_01ene06-max_des_sisf_dloc
]
# Las mismas bases, pero contra el corte del 01/07/2026: cada una se cruza contra
# el corte que le asigna DELTAS_VS_JULIO y lleva SUFIJO_DELTAS_VS_JULIO, igual que
# su marcador. Las columnas que vienen del corte llevan `_01jul26`:
#     output/Resultados_4_Generacion_Indicadores/reg_rn_01ene06-max_des_sisf_dloc_v2.parquet
BAJAS_VS_JULIO_A_EXPORTAR = [
    Slice("rn", "01ene06-max", estatus="des", subfiltro="sisf"),   # reg_rn_01ene06-max_des_sisf_dloc_v2
]

# Bases detrás de los marcadores de diferencia `<actual>-<corte>`: las altas y
# las bajas juntas, una fila por registro, con la columna
# `movimiento_derivada_rnpdno_25mar26` ("alta" / "baja"). Las altas menos las
# bajas dan el marcador, y no se escribe si no cuadra. Cada una se cruza contra
# el corte que le asigna DELTAS_VS_MARZO y lleva el nombre del marcador:
#     output/Resultados_4_Generacion_Indicadores/reg_rn_01ene06-max_des_sisf-reg_rn_25mar26_des_sisf.parquet
DIFERENCIAS_A_EXPORTAR = [
    Slice("rn", "01ene06-max", estatus="des", subfiltro="sisf"),   # reg_rn_01ene06-max_des_sisf-reg_rn_25mar26_des_sisf
]

# Las gráficas NO se declaran aquí: cada una vive en su propio bloque de la
# sección 4, con la rebanada, el nombre y TODOS sus parámetros escritos en la
# llamada. Se hizo así porque ninguna lámina se deja ajustar con los parámetros
# de la de al lado: un estilo compartido obligaba a tocar todas para mover una.
# Lo único que sigue aquí es lo que no es estilo: el desglose por dependencia y
# los nombres de los marcadores de la lámina 12.

# Series por entidad de las gráficas `gr_ent_*`: barras horizontales apiladas por la
# dependencia que levantó el reporte. Es lo que rotula la lámina ("por entidad
# y dependencia") y lo que desglosan sus tres recuadros laterales.
# El orden del diccionario fija el orden de apilado y, con él, el de `colores`.
DEPENDENCIAS = {"Fiscalía": "fis", "Comisión": "com", "Portal": "por"}

# ── Lámina 12 · indicios de vida ya formalizados como localización ───────────
# No es una rebanada del universo: la población sale del directorio congelado
# `config.DIRECTORIO_INDICIOS_FILE`, no del RNPDNO, y su "período" es la fecha
# de ese corte. Por eso no hay `Slice` y el conteo entra por `extra=`, igual
# que las diferencias contra el corte de marzo.
#
# El nombre de los marcadores se arma con la fecha del corte, así que cambiar
# `DIRECTORIO_INDICIOS_CORTE` en config los renombra todos a la vez.
MARCADOR_FMZ = f"reg_rn_{config.DIRECTORIO_INDICIOS_CORTE}-max_loc_fmz"
GRAFICA_FMZ  = f"gr_ent_rn_{config.DIRECTORIO_INDICIOS_CORTE}-max_loc_fmz"

# `_v2` es ese mismo conteo de formalizados SIN acotar a esas entidades: el
# total del país. `pct_` mide los formalizados de la lámina contra ESE total
# del país, o sea qué parte de los formalizados del directorio completo cae en
# las entidades de la Búsqueda Generalizada. Los tres salen de una sola pasada
# de `indicios_formalizados`.
MARCADOR_FMZ_PCT = f"pct_{MARCADOR_FMZ}"
MARCADOR_FMZ_NAC = f"{MARCADOR_FMZ}_v2"

# `_v3` es el complemento de `_v2`: los formalizados del RESTO del país, o sea
# los que quedan fuera de las entidades de la Búsqueda Generalizada.
#     reg_..._fmz  +  reg_..._fmz_v3  =  reg_..._fmz_v2
# La resta la hace `indicios_formalizados` (devuelve `resto`), no el cuaderno:
# así las tres cifras salen de la misma pasada y no pueden discrepar.
MARCADOR_FMZ_RESTO = f"{MARCADOR_FMZ}_v3"

# ── Lámina gemela · las 32 entidades federativas ────────────────────────────
# La misma lámina, pero sin acotar a las entidades de la Búsqueda Generalizada:
# la gráfica lleva las 32 entidades y `_v4` cuenta sus formalizados. No es el
# `_v2`: ése también cuenta a quien no tiene entidad ("Nacido en el extranjero").
#     reg_..._fmz  ≤  reg_..._fmz_v4  ≤  reg_..._fmz_v2
# Salen de la misma pasada de `indicios_formalizados` que los de arriba.
MARCADOR_FMZ_32 = f"{MARCADOR_FMZ}_v4"
GRAFICA_FMZ_32  = f"gr_32ent_rn_{config.DIRECTORIO_INDICIOS_CORTE}-max_loc_fmz"

# ── Fecha de inicio de los períodos que rotulan una lámina ──────────────────
# Estos marcadores se leen igual que `fha_rn_mes_anio`: la cola dice el formato
# y lo que va en medio dice DE QUÉ fecha se habla. Aquí no es un corte de datos
# sino el inicio de un período, así que lleva la llave de ese período:
#     fha_01oct24_mes_anio → "octubre de 2024"   (láminas 11 y 12)
#     fha_01ago26_mes_anio → "agosto de 2026"    (láminas 13 y 14)
# El valor no se teclea: sale del límite inferior del período en
# config.PERIODOS_TIEMPO, así que mover un corte reetiqueta su lámina sola.
PERIODOS_ROTULADOS = ["01oct24-max", "01ago26-max"]


## 3. Universo

Acierto de caché si el notebook 4 (o cualquier otra PPT con el mismo perfil de capa A)
ya lo construyó. Si desde entonces cambió `config.py`, el perfil, el código o algún insumo,
el caché no se usa y el universo se recalcula aquí.

In [ ]:
u = Universo.construir(cfg)

## 3.1 RNIP y MJ · el cruce de la última lámina

La última lámina no sale del universo. Su población es el archivo de **MJ y RIPS**
(`config.MJ_RIPS_FILE`). Lo único que aporta `u.rn` es el **estatus vigente** de cada `fub`, y de ahí
salen los cuatro subconjuntos que rotula la lámina.

Por eso no pasa por `Slice` ni entra en `CONTEOS`: es el mismo caso que los indicios
formalizados (`indicios_formalizados`), y sus conteos viajan por `extra=` en la celda siguiente.


In [ ]:
# `mr` = las dos hojas del archivo (MJ y RIPS) concatenadas SIN deduplicar —un
# `fub` que está en ambas son dos registros, que es lo que cuenta la lámina— y
# cruzadas contra `u.rn` por `fub`. Del RNPDNO sólo se trae lo que decide el
# reparto: `estatus_loc` (des/loc) y las banderas `is_cvid` / `is_svid`.
mr_dfs, mr_vars = cruzar_mj_rips(u.rn)

# Cada subconjunto lleva el nombre de SU marcador. Las dos parejas son
# particiones, así que sus porcentajes suman 100%:
#     reg_mr_min-max      = ..._des       + ..._loc
#     reg_mr_min-max_loc  = ..._loc_cvid  + ..._loc_svid
mr      = mr_dfs["reg_mr_min-max"]
mr_des  = mr_dfs["reg_mr_min-max_des"]
mr_loc  = mr_dfs["reg_mr_min-max_loc"]
mr_cvid = mr_dfs["reg_mr_min-max_loc_cvid"]
mr_svid = mr_dfs["reg_mr_min-max_loc_svid"]

mr.head(3)


In [ ]:
vars_ppt = u.indicadores(CONTEOS, porcentajes=PORCENTAJES)

# Las diferencias contra los cortes de marzo y julio se calculan DESPUÉS: necesitan los
# conteos ya resueltos, y se inyectan por `extra=` para que viajen en el mismo
# diccionario que consume PPTReportBuilder.
# Indicios del directorio que ya se formalizaron como localización (lámina 12).
# Devuelve los dos tabulados por entidad —el de las entidades de la Búsqueda
# Generalizada y el de las 32, uno por gráfica— y los conteos de sus marcadores.
tabulado_fmz, tabulado_fmz_32, fmz = indicios_formalizados(u.rn)

# Los repartos de las láminas 5 a 7, con las décimas ajustadas para que cada uno
# sume exactamente 100%: una pasada por entrada de PARTICIONES.
pct_particiones = {clave: valor
                   for sufijo, den, partes in PARTICIONES
                   for clave, valor in porcentajes_particion(u, partes, den, sufijo).items()}

vars_ppt = u.indicadores(
    CONTEOS, porcentajes=PORCENTAJES,
    extra={**pct_particiones,
           **diferencias_vs_corte(vars_ppt, DIFERENCIAS_VS_MARZO),
           **deltas_vs_corte(u, DELTAS_VS_MARZO),
           **diferencias_vs_corte(vars_ppt, DIFERENCIAS_VS_JULIO),
           **deltas_vs_corte(u, DELTAS_VS_JULIO, sufijo=SUFIJO_DELTAS_VS_JULIO),
           MARCADOR_FMZ:       fmz["total"],
           MARCADOR_FMZ_PCT:   fmz["pct"],
           MARCADOR_FMZ_NAC:   fmz["nacional"],
           MARCADOR_FMZ_RESTO: fmz["resto"],
           MARCADOR_FMZ_32:    fmz["total_32"],
           # Los cinco conteos de la última lámina, sus porcentajes y
           # `fha_mr_*`: ya vienen con el nombre de su marcador.
           **mr_vars,
           **{f"fha_{p.split('-')[0]}_mes_anio":
                  formatear_fecha_es(cfg.rango_periodo(p)[0], "mes_anio")
              for p in PERIODOS_ROTULADOS}},
)
dfs      = u.dataframes(PARA_GRAFICAS)

# materializar=True escribe cada rebanada a Parquet en cfg.resultados_dir.
exportados = u.dataframes(PARA_EXPORTAR, materializar=True)
# Las de PARA_EXPORTAR_CON_FECHA: Parquet + Excel, con la fecha de hoy en el nombre.
exportados_con_fecha = {s.nombre: exportar_con_fecha(u, s) for s in PARA_EXPORTAR_CON_FECHA}

for k, v in vars_ppt.items():
    print(f"  {k:<45} {v}")

## 3.2 Bajas y diferencias contra los cortes de marzo y julio

Los registros detrás de cada `_dloc` declarado en `BAJAS_A_EXPORTAR`: las filas del corte del
25/03/2026 cuyo `fub` ya no está en el indicador de hoy. No son una rebanada del universo —viven
en el Excel del corte—, por eso no entran en `PARA_EXPORTAR`. Los de `BAJAS_VS_JULIO_A_EXPORTAR`
son lo mismo contra el corte del 01/07/2026, y salen como `_dloc_v2`.

Se detiene si los `fub` no son exactamente los que cuenta el marcador `_dloc`, y desglosa por
qué salió cada registro: hoy localizado, ya no cumple el subfiltro, fecha fuera del período o
ya no está en el RNPDNO.

Después, la base de cada diferencia declarada en `DIFERENCIAS_A_EXPORTAR`: las altas (registros de hoy
que no estaban en el corte) y las bajas juntas, marcadas en `movimiento_derivada_rnpdno_25mar26`. Se
detiene si las altas menos las bajas no dan el valor del marcador.

In [ ]:
# Una base por indicador, en cfg.resultados_dir como `<indicador>_dloc.parquet`.
bajas = {f"{s.nombre}_dloc": bajas_vs_corte(u, s, DELTAS_VS_MARZO[s], materializar=True)
         for s in BAJAS_A_EXPORTAR}
# Las de julio, como `<indicador>_dloc_v2.parquet`.
bajas |= {f"{s.nombre}_dloc{SUFIJO_DELTAS_VS_JULIO}":
              bajas_vs_corte(u, s, DELTAS_VS_JULIO[s], sufijo=SUFIJO_DELTAS_VS_JULIO,
                             materializar=True)
          for s in BAJAS_VS_JULIO_A_EXPORTAR}

# Una base por marcador de diferencia, en cfg.resultados_dir como `<indicador>-<corte>.parquet`.
diferencias = {f"{s.nombre}-{DELTAS_VS_MARZO[s]}":
                   movimientos_vs_corte(u, s, DELTAS_VS_MARZO[s], materializar=True)
               for s in DIFERENCIAS_A_EXPORTAR}

## 4. Gráficas

Una gráfica, un bloque, con sus parámetros escritos en la llamada: ajustar una lámina no
toca a las demás. Lo único compartido es la preparación de los datos.

In [ ]:
df_match = dfs["reg_rn-si-ba_01oct24-max_des"]

# Ejemplo: distribución por entidad de los desaparecidos del sexenio con carpeta en BANCI
serie = (df_match.groupby(u.columna("rn", "entidad"), observed=True)
                 .size()
                 .sort_values(ascending=False)
                 .head(10))
print(serie)

# `exportar_grafica` (1_funciones/funciones.py) tiene "output/<nombre>"
# codificado y relativo al directorio de trabajo, así que los PNG y SVG caen
# en notebooks/output/ sin importar lo que se declare aquí. RUTA_GRAFICAS
# apunta a esa ruta real para que el armado de la PPT encuentre las imágenes.
RUTA_GRAFICAS = Path("output")
RUTA_GRAFICAS.mkdir(parents=True, exist_ok=True)

# Una rebanada sin registros (p. ej. un período corto que aún no tiene casos en
# alguna rama) haría tronar la gráfica y, con ella, toda la PPT. Envueltas, en
# vez de tronar escriben en su PNG un aviso "Sin registros en el período" con
# las medidas de la caja. No se salta la gráfica: el PNG de la corrida anterior
# seguiría en disco y entraría en la plantilla con datos de otro día.
barras_verticales   = tolerar_sin_datos(barras_verticales,   RUTA_GRAFICAS)
barras_horizontales = tolerar_sin_datos(barras_horizontales, RUTA_GRAFICAS)

# ─────────────────────────────────────────────────────────────────────────────
# Una gráfica, un bloque. Cada llamada lleva TODOS sus parámetros escritos: para
# mover una lámina se edita SU bloque y ninguna otra cambia. Lo único compartido
# es la preparación de los datos —`serie_mensual`, `tabulado_dependencia`—, que
# no es estilo: es de dónde salen las barras.
#
# Para agregar una gráfica se copia el bloque más parecido, se le cambia la
# rebanada y el nombre, y se ajustan sus parámetros. `registrar` la apunta para
# la sección 5, que es la que la inserta en la plantilla.
# ─────────────────────────────────────────────────────────────────────────────
GRAFICAS_GENERADAS = []
GRAFICAS_SIN_DATOS = []   # las que llevan el aviso; se listan en la sección 5


def registrar(nombre, tabla):
    """Apunta la gráfica para la sección 5 y muestra el PNG que se guardó.

    Se muestra el archivo y no la figura viva porque el PNG es exactamente lo que
    entra en la plantilla. El SVG que también escribe `exportar_grafica` queda en
    disco pero no se usa ni aquí ni en el PPTX.
    """
    total = tabla.select_dtypes("number").to_numpy().sum()
    print(f"  📊 {nombre}: {total:,} registros en {len(tabla)} filas")
    GRAFICAS_GENERADAS.append(nombre)
    if sin_registros(tabla):
        GRAFICAS_SIN_DATOS.append(nombre)
    display(Image(filename=str(RUTA_GRAFICAS / f"{nombre}.png")))


def serie_mensual(s):
    """Registros por mes de una rebanada.

    `completar_meses=True` rellena con cero los meses sin registros: sin eso, un
    hueco desaparece del eje y la serie aparenta continuidad donde no la hay.
    `desde`/`hasta` llevan ese relleno hasta los extremos del PERÍODO y no sólo
    entre el primer y el último registro: a `sfval_cfvac` se le acaban en julio
    de 2026 y sin esto su eje terminaba ahí, dos meses antes que las demás.
    """
    desde, hasta = u.rango_fechas(s)
    return agrupar_por_mes(u.slice(s), u.columna(s, "fecha"), completar_meses=True,
                           desde=desde, hasta=hasta)


def serie_mensual_apilada(series):
    """Registros por mes de varias rebanadas, una columna por rebanada.

    Es la tabla que espera `barras_verticales` para apilar: la primera columna es
    el mes y cada una de las siguientes es un segmento de la barra, en el orden
    del diccionario (el primero queda abajo). Rebanadas de la misma base y el
    mismo período reciben de `serie_mensual` la misma rejilla de meses; el
    `fillna(0)` sólo cubre el caso de que no la compartieran.
    """
    return (pd.concat({etq: serie_mensual(s).set_index("mes")["registros"]
                       for etq, s in series.items()}, axis=1)
              .fillna(0).astype(int)
              .rename_axis("mes").reset_index())


def tabulado_dependencia(s):
    """Conteo por entidad y dependencia: las tres series apiladas de `gr_ent_*`.

    Sumar las banderas booleanas da directamente el conteo por dependencia, y
    `observed=True` deja fuera las entidades sin registros en la rebanada. El
    orden de `DEPENDENCIAS` fija el de las columnas y, con él, el de `colores`.
    """
    d = u.slice(s)
    return (pd.DataFrame({"entidad": d[u.columna(s, "entidad")],
                          **{etq: d[u.columna_bandera(s, tok)] for etq, tok in DEPENDENCIAS.items()}})
            .groupby("entidad", observed=True).sum().reset_index())


# ── gr_mes · con match en BANCI ──────────────────────────────────────────────
# `dimensiones_placeholder` da el tamaño de SU cuadro en la plantilla: el motor
# inserta la figura forzando ancho y alto de la caja, así que generarla con otra
# proporción la deformaría. `mostrar=False` evita que matplotlib pinte la figura
# en vivo; lo que se ve abajo es el PNG ya guardado.
nombre = "gr_mes_rn-si-ba_01oct24-max_des"
tabla = serie_mensual(
    Slice("rn-si-ba", "01oct24-max", estatus="des"))
ancho_cm, alto_cm = dimensiones_placeholder(PLANTILLA, nombre)
barras_verticales(
    tabla, nombre=nombre, mostrar=False,
    ancho=ancho_cm, alto=alto_cm,
    colores=["#af5967ff"],
    tam_valor_ejex=7,
    tam_valor_ejey=11,
    valor_barra=False,
    tam_valor_capsu=6.8,
    ajustar_pos_capsu=0.03,
    peso_borde_capsu=0.8,
    ordenar="x",             # el eje X son fechas: `ordenar="x"` las pone cronológicas
)
registrar(nombre, tabla)

# ── gr_mes · sin match en BANCI, con folio válido ────────────────────────────
nombre = "gr_mes_rn-no-ba_01oct24-max_des_cfval"
tabla = serie_mensual(
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="cfval"))
ancho_cm, alto_cm = dimensiones_placeholder(PLANTILLA, nombre)
barras_verticales(
    tabla, nombre=nombre, mostrar=False,
    ancho=ancho_cm, alto=alto_cm,
    colores=["#9b2347ff"],
    tam_valor_ejex=7,
    tam_valor_ejey=11,
    valor_barra=False,
    tam_valor_capsu=6.8,
    ajustar_pos_capsu=0.03,
    peso_borde_capsu=0.8,
    ordenar="x",
)
registrar(nombre, tabla)

# ── gr_mes · sin match en BANCI, sin folio válido (sfval completo) ───────────
# Las dos de abajo juntas: folio inválido + campo vacío. Es la lámina "Sin
# carpeta de investigación", que va antes de su desglose en dos.
nombre = "gr_mes_rn-no-ba_01oct24-max_des_sfval"
tabla = serie_mensual(
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval"))
ancho_cm, alto_cm = dimensiones_placeholder(PLANTILLA, nombre)
barras_verticales(
    tabla, nombre=nombre, mostrar=False,
    ancho=ancho_cm, alto=alto_cm,
    colores=["#595959ff"],
    tam_valor_ejex=7,
    tam_valor_ejey=11,
    valor_barra=False,
    tam_valor_capsu=6.8,
    ajustar_pos_capsu=0.03,
    peso_borde_capsu=0.8,
    ordenar="x",
)
registrar(nombre, tabla)

# ── gr_mes · sin match en BANCI, folio inválido ──────────────────────────────
nombre = "gr_mes_rn-no-ba_01oct24-max_des_sfval_sfvac"
tabla = serie_mensual(
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval_sfvac"))
ancho_cm, alto_cm = dimensiones_placeholder(PLANTILLA, nombre)
barras_verticales(
    tabla, nombre=nombre, mostrar=False,
    ancho=ancho_cm, alto=alto_cm,
    colores=["#757575ff"],
    tam_valor_ejex=7,
    tam_valor_ejey=11,
    valor_barra=False,
    tam_valor_capsu=6.8,
    ajustar_pos_capsu=0.03,
    peso_borde_capsu=0.8,
    ordenar="x",
)
registrar(nombre, tabla)

# ── gr_mes · sin match en BANCI, campo de folio vacío ────────────────────────
# La otra rama de `sfval`. Misma lectura mensual que las dos de arriba; el color
# es el de la etiqueta de SU lámina.
nombre = "gr_mes_rn-no-ba_01oct24-max_des_sfval_cfvac"
tabla = serie_mensual(
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval_cfvac"))
ancho_cm, alto_cm = dimensiones_placeholder(PLANTILLA, nombre)
barras_verticales(
    tabla, nombre=nombre, mostrar=False,
    ancho=ancho_cm, alto=alto_cm,
    colores=["#888888ff"],
    tam_valor_ejex=7,
    tam_valor_ejey=11,
    valor_barra=False,
    tam_valor_capsu=6.8,
    ajustar_pos_capsu=0.03,
    peso_borde_capsu=0.8,
    ordenar="x",
)
registrar(nombre, tabla)

# ── gr_mes · sexenio, las tres ramas de información suficiente, apiladas ─────
# Una barra por mes desde oct 2024, partida en las tres rebanadas que suman los
# desaparecidos del sexenio: sisf + cisf_cind + cisf_sind = reg_rn_01oct24-max_des.
# La lámina no lleva leyenda: sus recuadros laterales hacen de leyenda, en el
# mismo orden de arriba a abajo que la pila y con los mismos colores.
nombre = "gr_mes_rn_01oct24-max_des_sisf-cind-sind"
tabla = serie_mensual_apilada({
    "Datos insuficientes": Slice("rn", "01oct24-max", estatus="des", subfiltro="sisf"),
    "Con actividades":     Slice("rn", "01oct24-max", estatus="des", subfiltro="cisf_cind"),
    "Sin actividad":       Slice("rn", "01oct24-max", estatus="des", subfiltro="cisf_sind"),
})
ancho_cm, alto_cm = dimensiones_placeholder(PLANTILLA, nombre)
barras_verticales(
    tabla, nombre=nombre, mostrar=False,
    ancho=ancho_cm, alto=alto_cm,
    colores=["#ed7d31ff", "#2a7473ff", "#56408fff"],
    tam_valor_ejex=7,
    tam_valor_ejey=11,
    tam_valor_capsu=6.8,
    ajustar_pos_capsu=0.03,
    peso_borde_capsu=0.8,
    ordenar="x",
    valor_barra=True,
    tam_valor_barra=7.3,
    areamin = 30,
    ajustar_sep_valor_barra=1.5,
    ajustar_pos_capsu_areamin = 0.04,
)
registrar(nombre, tabla)

# ── gr_ent · sexenio (desde oct 2024), con folio válido ─────────────────────
# `barras_horizontales` no tiene interruptor `mostrar`: siempre llama a
# plt.show(). Se captura esa salida para que el cuaderno muestre el PNG guardado
# —el mismo archivo que entra en la plantilla— y no la figura viva.
# El tamaño no sale de la plantilla: las cuatro cajas `gr_ent_*` miden 18.03 x
# 10.82 cm, y 20 x 12 es esa misma proporción (1.667) con más margen para el texto.
nombre = "gr_ent_rn-no-ba_01oct24-max_des_cfval"
tabla = tabulado_dependencia(
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="cfval"))
with capture_output(stdout=False, stderr=False, display=True):
    barras_horizontales(
        tabla, nombre=nombre,
        ancho=20, alto=12,
        colores=["#4f3130", "#e44d03", "#de639b"],
        tam_valor_ejex=12,
        tam_valor_ejey=10,
        tam_valor_barra=8,
        tam_valor_capsu=7,
        ajustar_pos_capsu=0.01,
        peso_borde_capsu=0.8,
        areamin=30,
        leyenda=False,       # la leyenda son los recuadros de la plantilla
        ajustar_sep_valor_barra=1.3,
        )
registrar(nombre, tabla)

# ── gr_ent · sexenio (desde oct 2024), folio inválido ───────────────────────
nombre = "gr_ent_rn-no-ba_01oct24-max_des_sfval_sfvac"
tabla = tabulado_dependencia(
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval_sfvac"))
with capture_output(stdout=False, stderr=False, display=True):
    barras_horizontales(
        tabla, nombre=nombre,
        ancho=20, alto=12,
        colores=["#4f3130", "#e44d03", "#de639b"],
        tam_valor_ejex=12,
        tam_valor_ejey=10,
        tam_valor_barra=8,
        tam_valor_capsu=7,
        ajustar_pos_capsu=0.01,
        peso_borde_capsu=0.8,
        areamin=30,
        leyenda=False,
        ajustar_sep_valor_barra=1.3,
    )
registrar(nombre, tabla)

# ── gr_ent · desde agosto 2026, con folio válido ────────────────────────────
nombre = "gr_ent_rn-no-ba_01ago26-max_des_cfval"
tabla = tabulado_dependencia(
    Slice("rn-no-ba", "01ago26-max", estatus="des", subfiltro="cfval"))
with capture_output(stdout=False, stderr=False, display=True):
    barras_horizontales(
        tabla, nombre=nombre,
        ancho=20, alto=12,
        colores=["#4f3130", "#e44d03", "#de639b"],
        tam_valor_ejex=12,
        tam_valor_ejey=10,
        tam_valor_barra=8,
        tam_valor_capsu=7,
        ajustar_pos_capsu=0.01,
        peso_borde_capsu=0.8,
        areamin=0,
        leyenda=False,
    )
registrar(nombre, tabla)

# ── gr_ent · desde agosto 2026, folio inválido ──────────────────────────────
nombre = "gr_ent_rn-no-ba_01ago26-max_des_sfval_sfvac"
tabla = tabulado_dependencia(
    Slice("rn-no-ba", "01ago26-max", estatus="des", subfiltro="sfval_sfvac"))
with capture_output(stdout=False, stderr=False, display=True):
    barras_horizontales(
        tabla, nombre=nombre,
        ancho=20, alto=12,
        colores=["#4f3130", "#e44d03", "#de639b"],
        tam_valor_ejex=12,
        tam_valor_ejey=12,
        tam_valor_barra=11,
        tam_valor_capsu=13,
        ajustar_pos_capsu=0.03,
        peso_borde_capsu=1.3,
        areamin=0,
        valor_barra=False,
        leyenda=False,
        ajustar_sep_valor_barra=10,
    )
registrar(nombre, tabla)

# ── gr_ent · lámina 12 · indicios formalizados, por entidad ──────────────────
# Barras horizontales apiladas: cuántos siguen desaparecidos y cuántos ya se
# localizaron, por entidad. `tabulado_fmz` (sección 3) ya viene ordenado y
# acotado a DIRECTORIO_INDICIOS_ESTADOS; sus dos columnas de conteo son las dos
# series apiladas, así que aquí no hay preparación de datos que hacer.
print(f"  📇 {fmz['total']:,} formalizados en las entidades de la lámina")
with capture_output(stdout=False, stderr=False, display=True):
    barras_horizontales(
        tabulado_fmz, nombre=GRAFICA_FMZ,
        ancho=20, alto=12,
        colores=["#32485f", "#b3c5bf"],
        tam_valor_ejex=14,
        tam_valor_ejey=9,
        tam_valor_barra=7,
        tam_valor_capsu=11,
        valor_capsu=False,
        ajustar_pos_capsu=0.04,
        ajustar_pos_capsu_areamin=0.05,
        ajustar_sep_valor_barra=2.5,
        areamin=190,
        leyenda=False,
    )
registrar(GRAFICA_FMZ, tabulado_fmz)

# ── gr_32ent · indicios formalizados, las 32 entidades ──────────────────────
# La gemela de la anterior, con las 32 entidades federativas en vez de sólo las
# de la Búsqueda Generalizada. Su caja en la plantilla mide lo mismo, así que
# arranca con los mismos parámetros; 32 barras caben en ese alto.
print(f"  📇 {fmz['total_32']:,} formalizados en las 32 entidades")
with capture_output(stdout=False, stderr=False, display=True):
    barras_horizontales(
        tabulado_fmz_32, nombre=GRAFICA_FMZ_32,
        ancho=20, alto=12,
        colores=["#32485f", "#b3c5bf"],
        tam_valor_ejex=14,
        tam_valor_ejey=9,
        tam_valor_barra=7,
        tam_valor_capsu=11,
        valor_capsu=False,
        ajustar_pos_capsu=0.04,
        ajustar_pos_capsu_areamin=0.05,
        ajustar_sep_valor_barra=2.5,
        areamin=190,
        leyenda=False,
    )
registrar(GRAFICA_FMZ_32, tabulado_fmz_32)


## 5. Armado de la presentación

In [ ]:
SALIDA.parent.mkdir(parents=True, exist_ok=True)

if PLANTILLA.exists():
    builder = PPTReportBuilder(PLANTILLA)
    builder.add_text_variables(vars_ppt)
    # Las claves van sin llaves: en la plantilla el marcador es {{reg_rn_01oct24-max_des}}.
    # Cada gráfica sustituye al cuadro de texto que lleva su marcador, heredando
    # su posición y sus medidas.
    builder.add_image_variables({
        nombre: RUTA_GRAFICAS / f"{nombre}.png" for nombre in GRAFICAS_GENERADAS
    })

    # render() sustituye textos e imágenes, guarda el PPTX y devuelve el reporte
    # de qué marcadores se encontraron y cuáles no.
    reporte = builder.render(SALIDA)

    faltantes = [k for k, v in reporte["text_report"].items() if v == 0]
    if faltantes:
        print(f"\n⚠️ {len(faltantes)} variables sin marcador en la plantilla: {faltantes}")
    sin_imagen = [k for k, v in reporte["image_report"].items() if v == 0]
    if sin_imagen:
        print(f"⚠️ {len(sin_imagen)} gráficas sin placeholder en la plantilla: {sin_imagen}")
    if GRAFICAS_SIN_DATOS:
        print(f"⚠️ {len(GRAFICAS_SIN_DATOS)} gráficas sin registros; llevan el aviso "
              f"en su lugar: {GRAFICAS_SIN_DATOS}")
else:
    print(f"⚠️ Falta la plantilla {PLANTILLA}. Indicadores l istos en `vars_ppt`.")

## 6. (Opcional) Variante estatal

Las 32 entidades salen del **mismo** universo ya cargado: son máscaras booleanas, no
32 corridas de matching. Éste es el cambio que hizo posible mover `estado` fuera de la
etapa de matching.

In [ ]:
from src.slices import expandir_entidades

resumen_estatal = pd.DataFrame([
    {"entidad": s.entidad_texto, "registros": u.contar(s)}
    for s in expandir_entidades(Slice("rn", "01oct24-max", estatus="des"))
]).sort_values("registros", ascending=False)

resumen_estatal.head(10)

## 7. Base maestra para reconstruir los indicadores

Exporta un CSV unificado con los registros y campos necesarios para volver a calcular los conteos, porcentajes, cortes de fecha, desgloses por entidad, métodos de coincidencia, MJ/RIPS e indicios usados en esta presentación. Incluye al final filas de referencia con el valor de cada marcador calculado (`record_type = indicador_calculado`).

El archivo contiene FUB como llave de cruce; resguárdalo como información sensible. No incluye nombres, CURP, domicilios ni teléfonos.

In [ ]:
# Normaliza las fuentes del reporte a una tabla larga, manteniendo indicadores booleanos
# y las llaves/fechas necesarias para reconstruir máscaras, series y cortes.
COLUMNAS_MAESTRAS = [
    "record_type", "fuente", "fuente_detalle", "fub", "fecha_referencia",
    "entidad_clave", "entidad", "estatus_loc", "clasificacion_des", "tipo_match",
    "indicador_nombre", "indicador_valor", "valor_numerico",
    "estado_de_desaparicion", "todavia_desaparecidos", "localizados",
]


def _columna_maestra(df, candidatas):
    return next((c for c in candidatas if c in df.columns), None)


def _preparar_registros_maestros(df, fuente, fub_col, fecha_col=None,
                                 entidad_col=None, clave_ent_col=None,
                                 estatus_col=None, clasificacion_col=None,
                                 tipo_match_col=None):
    """Proyecta datos analíticos sin exponer identidad directa."""
    salida = pd.DataFrame(index=df.index)
    salida["record_type"] = "registro"
    salida["fuente"] = fuente
    salida["fuente_detalle"] = fuente
    salida["fub"] = df[fub_col].astype("string").str.strip() if fub_col in df else pd.NA
    if fecha_col and fecha_col in df:
        salida["fecha_referencia"] = pd.to_datetime(df[fecha_col], errors="coerce").dt.strftime("%Y-%m-%d")
    if entidad_col and entidad_col in df:
        salida["entidad"] = df[entidad_col].astype("string")
    if clave_ent_col and clave_ent_col in df:
        salida["entidad_clave"] = pd.to_numeric(df[clave_ent_col], errors="coerce").astype("Int64")
    if estatus_col and estatus_col in df:
        salida["estatus_loc"] = df[estatus_col].astype("string")
    if clasificacion_col and clasificacion_col in df:
        salida["clasificacion_des"] = df[clasificacion_col].astype("string")
    if tipo_match_col and tipo_match_col in df:
        salida["tipo_match"] = df[tipo_match_col].astype("string")
    # Las banderas de negocio son las columnas que hacen posible rearmar subfiltros.
    for col in df.columns:
        if col.startswith(("is_", "in_banci_", "in_rnpdno_")):
            salida[col] = df[col].astype("boolean")
    return salida.reset_index(drop=True)


# RNPDNO y BANCI: universos completos, no solo las rebanadas visibles en la PPT.
_base_rn_maestra = _preparar_registros_maestros(
    u.rn, "universo_rnpdno", "fub_rnpdno",
    fecha_col=u.columna("rn", "fecha"),
    entidad_col=u.columna("rn", "entidad"),
    clave_ent_col=u.columna("rn", "cve_ent"),
    estatus_col=u.columna("rn", "estatus_loc"),
)
_base_ba_maestra = _preparar_registros_maestros(
    u.ba, "universo_banci", "fub_derivada_rnpdno_banci",
    fecha_col=u.columna("ba", "fecha"),
    entidad_col=u.columna("ba", "entidad"),
    clave_ent_col=u.columna("ba", "cve_ent"),
    estatus_col=u.columna("ba", "estatus_loc"),
    tipo_match_col="tipo_match_derivada_rnpdno_banci",
)

# La hoja MJ/RIPS es un universo externo independiente; no se deduplica entre hojas.
_mr_completo = mr_dfs["reg_mr_min-max"].copy()
if "fub" not in _mr_completo.columns:
    raise KeyError("La base MJ/RIPS no contiene la llave fub requerida.")
_base_mr_maestra = _preparar_registros_maestros(
    _mr_completo, "mj_rips", "fub",
    fecha_col=_columna_maestra(_mr_completo, ["fecha_hechos", "fecha_evento"]),
    entidad_col=_columna_maestra(_mr_completo, ["estado", "entidad"]),
    estatus_col=_columna_maestra(_mr_completo, ["estatus_loc"]),
)
if "tipo" in _mr_completo:
    _base_mr_maestra["fuente_detalle"] = _mr_completo["tipo"].astype("string").to_numpy()
if "is_cvid" in _mr_completo:
    _base_mr_maestra["is_cvid"] = _mr_completo["is_cvid"].astype("boolean").to_numpy()
if "is_svid" in _mr_completo:
    _base_mr_maestra["is_svid"] = _mr_completo["is_svid"].astype("boolean").to_numpy()

# El directorio congelado se cruza con la clasificación RNPDNO vigente para
# reconstruir los conteos de formalizados (24 entidades, 32 entidades y país).
_dir_path = config.DIRECTORIO_INDICIOS_FILE
if not _dir_path.is_file():
    raise FileNotFoundError(f"Falta el directorio de indicios configurado: {_dir_path}")
_dir_raw = pd.read_excel(_dir_path, sheet_name=config.DIRECTORIO_INDICIOS_SHEET)
_dir_cols = {}
for _destino, _aliases in config.DIRECTORIO_INDICIOS_COLUMN_MAPPING.items():
    _alias_list = [_aliases] if isinstance(_aliases, str) else list(_aliases)
    _real = next((a for a in _alias_list if a in _dir_raw.columns), None)
    if _real is None:
        raise KeyError(f"El directorio de indicios no contiene {_destino}; columnas: {list(_dir_raw.columns)}")
    _dir_cols[_destino] = _real
_dir = _dir_raw[[_dir_cols["fub"], _dir_cols["estado_de_desaparicion"]]].rename(
    columns={_dir_cols["fub"]: "fub", _dir_cols["estado_de_desaparicion"]: "estado_de_desaparicion"}
).copy()
_dir["fub"] = _dir["fub"].astype("string").str.upper().str.strip()
_dir["estado_de_desaparicion"] = (
    _dir["estado_de_desaparicion"].fillna(config.DIRECTORIO_INDICIOS_SIN_ESTADO)
    .replace(config.DIRECTORIO_INDICIOS_RENOMBRES)
)
_clas_rn = u.rn[["fub_rnpdno", "clasificacion_des_derivada_rnpdno"]].rename(
    columns={"fub_rnpdno": "fub", "clasificacion_des_derivada_rnpdno": "clasificacion_des"}
).copy()
_clas_rn["fub"] = _clas_rn["fub"].astype("string").str.upper().str.strip()
_dir = _dir.merge(_clas_rn.drop_duplicates("fub"), on="fub", how="left")
_dir["estatus_loc"] = np.where(_dir["clasificacion_des"].eq("Localizados"), "loc", "des")
_base_dir_maestra = pd.DataFrame({
    "record_type": "registro",
    "fuente": "directorio_indicios",
    "fuente_detalle": "corte_" + str(config.DIRECTORIO_INDICIOS_CORTE),
    "fub": _dir["fub"],
    "estatus_loc": _dir["estatus_loc"],
    "clasificacion_des": _dir["clasificacion_des"],
    "estado_de_desaparicion": _dir["estado_de_desaparicion"],
})

# FUBs de los seis cortes congelados: son la base para recalcular altas/bajas y diferencias.
_bases_cortes_maestras = []
for _marcador, (_ruta_corte, _col_fub_corte) in config.CORTES_REFERENCIA.items():
    if not _ruta_corte.is_file():
        raise FileNotFoundError(f"Falta el corte de referencia {_marcador}: {_ruta_corte}")
    _encabezado = pd.read_excel(_ruta_corte, nrows=0).columns
    _normalizar = lambda x: "".join(ch for ch in unicodedata.normalize("NFD", str(x)) if not unicodedata.combining(ch)).lower().replace("_", "").replace(" ", "").replace(".", "")
    _real_fub = next((c for c in _encabezado if _normalizar(c) == _normalizar(_col_fub_corte)), None)
    if _real_fub is None:
        raise KeyError(f"{_ruta_corte.name}: no se encontró columna {_col_fub_corte}")
    _fubs_corte = pd.read_excel(_ruta_corte, usecols=[_real_fub], dtype={_real_fub: "string"})[_real_fub]
    _df_corte = pd.DataFrame({
        "record_type": "registro",
        "fuente": "corte_referencia",
        "fuente_detalle": _marcador,
        "fub": _fubs_corte.astype("string").str.strip(),
        "indicador_nombre": _marcador,
    })
    _bases_cortes_maestras.append(_df_corte)

# Filas de control: cada marcador que el notebook calcula queda incluido para conciliar
# la reconstrucción independiente contra la salida real de la presentación.
_base_indicadores = pd.DataFrame({
    "record_type": "indicador_calculado",
    "fuente": "reporte_diario_6",
    "fuente_detalle": "vars_ppt",
    "indicador_nombre": list(vars_ppt.keys()),
    "indicador_valor": [str(v) for v in vars_ppt.values()],
})
_base_indicadores["valor_numerico"] = pd.to_numeric(_base_indicadores["indicador_valor"], errors="coerce")

# El tabulado de indicios también se conserva agregado para reproducir sus gráficos.
def _filas_tabulado_indicios(tabla, detalle):
    out = pd.DataFrame({
        "record_type": "resumen_entidad",
        "fuente": "directorio_indicios",
        "fuente_detalle": detalle,
        "entidad": tabla["estado_de_desaparicion"].astype("string"),
        "estado_de_desaparicion": tabla["estado_de_desaparicion"].astype("string"),
        "todavia_desaparecidos": tabla["todavia_desaparecidos"],
        "localizados": tabla["localizados"],
    })
    return out

_base_tabulados_indicios = pd.concat([
    _filas_tabulado_indicios(tabulado_fmz, "entidades_busqueda_generalizada"),
    _filas_tabulado_indicios(tabulado_fmz_32, "32_entidades"),
], ignore_index=True)

# Concatenación con esquema amplio: columnas booleanas distintivas de RN/BANCI se
# preservan y el mismo CSV incluye las fuentes auxiliares y valores de referencia.
base_maestra_reporte = pd.concat([
    _base_rn_maestra, _base_ba_maestra, _base_mr_maestra, _base_dir_maestra,
    *_bases_cortes_maestras, _base_tabulados_indicios, _base_indicadores,
], ignore_index=True, sort=False)

RUTA_BASE_MAESTRA = config.OUTPUT_DIR / "Base_Maestra_Reporte_Diario.csv"
RUTA_BASE_MAESTRA.parent.mkdir(parents=True, exist_ok=True)
base_maestra_reporte.to_csv(RUTA_BASE_MAESTRA, index=False, encoding="utf-8-sig")

# Relectura/controles: esquema, tamaño de fuentes principales, FUBs de corte y catálogo de marcadores.
_base_maestra_check = pd.read_csv(RUTA_BASE_MAESTRA, encoding="utf-8-sig", low_memory=False)
assert len(_base_maestra_check) == len(base_maestra_reporte)
assert set(_base_maestra_check["record_type"].dropna()) == {
    "registro", "resumen_entidad", "indicador_calculado"
}
assert int((_base_maestra_check["fuente"] == "universo_rnpdno").sum()) == len(u.rn)
assert int((_base_maestra_check["fuente"] == "universo_banci").sum()) == len(u.ba)
assert int((_base_maestra_check["record_type"] == "indicador_calculado").sum()) == len(vars_ppt)
print("Base maestra CSV exportada:", RUTA_BASE_MAESTRA)
print(f"Filas: {len(_base_maestra_check):,} | columnas: {len(_base_maestra_check.columns)} | indicadores control: {len(vars_ppt)}")
print("Registros por fuente:")
print(_base_maestra_check.groupby(["record_type", "fuente"], dropna=False).size().to_string())